# Kết quả mô hình

## 1. Đọc kết quả

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path('/kaggle/working/well-log-imputation')
RUN = Path('/kaggle/working/well-log-results-final/TEN_LAN_CHAY')
FIG_DIR = ROOT / 'results' / 'visualizations'
FIG_DIR.mkdir(parents=True, exist_ok=True)
SPLIT = 'test'
SEED = 912
SMOOTH_WINDOW = 5

SCENARIOS = ['Single', 'Block-20', 'Block-100', 'Entire-Log']
SCENARIO_LABELS = {
    'Single': 'Thiếu\nmột điểm', 'Block-20': 'Thiếu đoạn\n20 điểm',
    'Block-100': 'Thiếu đoạn\n100 điểm', 'Entire-Log': 'Thiếu\ntoàn bộ log',
}
LABELS = {'locf': 'LOCF', 'xgboost': 'XGBoost', 'brits': 'BRITS',
          'saits': 'SAITS', 'conv_saits': 'Conv-SAITS'}
COLORS = {'locf': '#1f77a4', 'xgboost': '#b878a2', 'brits': '#bc8220',
          'saits': '#4477aa', 'conv_saits': '#228866'}
METRIC_LABELS = {'mae': 'Sai số tuyệt đối trung bình',
                 'rmse': 'Căn sai số bình phương trung bình'}
UNIT_LABELS = {'normalized': 'thang chuẩn hóa', 'original': 'đơn vị gốc'}
NEURAL_MODELS = ['brits', 'saits', 'conv_saits']

summary = pd.read_csv(RUN / 'summary.csv').query('split == @SPLIT')
detailed = pd.read_csv(RUN / 'summary_detailed.csv').query('split == @SPLIT')
history = pd.read_csv(RUN / 'training_history.csv')
runtime = pd.read_csv(RUN / 'runtime_by_seed.csv')
manifest = json.loads((RUN / 'experiment_results.json').read_text())
runs = pd.DataFrame(manifest['runs'])
MODELS = [model for model in LABELS if model in summary['model'].unique()]
logs = detailed[detailed['level'] == 'log']
LOGS = logs['group'].unique().tolist()
print('Kết quả:', RUN)


## 2. Hàm vẽ biểu đồ

In [ ]:
def save_figure(fig, name):
    fig.savefig(FIG_DIR / f'{name}.png', dpi=300)
    plt.show()
    plt.close(fig)


def bar_chart(frame, column, ax, ylabel):
    table = frame.pivot(index='scenario', columns='model', values=column)
    table = table.reindex(index=SCENARIOS, columns=MODELS)
    table.rename(columns=LABELS).plot.bar(
        ax=ax, color=[COLORS[model] for model in MODELS], rot=0, width=0.8,
    )
    ax.set(xlabel='Kịch bản thiếu dữ liệu', ylabel=ylabel)
    ax.get_legend().set_title('Mô hình')
    ax.set_xticklabels([SCENARIO_LABELS[scenario] for scenario in SCENARIOS], fontsize=9)
    ax.grid(axis='y', alpha=0.25)
    ax.set_axisbelow(True)


def plot_logs(metric, units):
    frame = logs[logs['units'] == units]
    fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharey=units == 'normalized')
    for ax, log in zip(axes.flat, LOGS):
        bar_chart(frame[frame['group'] == log], f'{metric}_mean', ax, metric.upper())
        ax.set_title(log)
        ax.get_legend().remove()
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=len(MODELS), frameon=False)
    fig.suptitle(f'{METRIC_LABELS[metric]} ({UNIT_LABELS[units]})', y=0.94)
    fig.tight_layout(rect=(0, 0, 1, 0.91))
    save_figure(fig, f'{SPLIT}_{metric}_{units}')


## 3. MAE và RMSE theo từng log

In [ ]:
for metric in ['mae', 'rmse']:
    for units in ['normalized', 'original']:
        plot_logs(metric, units)


## 4. MAPE trung bình — GR, RHOB, DTC

In [ ]:
mape = logs.query("units == 'original' and group in ['GR', 'RHOB', 'DTC']")
mape = mape.groupby(['model', 'scenario'], as_index=False)['mape_mean'].mean()
fig, ax = plt.subplots(figsize=(10, 5))
bar_chart(mape, 'mape_mean', ax, 'MAPE trung bình (%)')
ax.set_title('Sai số phần trăm tuyệt đối trung bình — GR, RHOB, DTC')
fig.tight_layout()
save_figure(fig, f'{SPLIT}_mape')


## 5. Bảng MAE và RMSE tổng thể — thang chuẩn hóa


In [ ]:
table = summary.pivot(index='scenario', columns='model', values=['mae_mean', 'rmse_mean'])
table = table.reindex(SCENARIOS).rename(columns=LABELS, level='model')
table = table.rename(columns={'mae_mean': 'MAE trung bình', 'rmse_mean': 'RMSE trung bình'}, level=0)
table = table.rename(index={key: value.replace('\n', ' ') for key, value in SCENARIO_LABELS.items()})
table.index.name = 'Kịch bản thiếu dữ liệu'
table.to_csv(FIG_DIR / f'{SPLIT}_overall_metrics.csv')
display(table.round(4))


## 6. Hội tụ theo vòng huấn luyện

In [ ]:
curves = history[history['model'].isin(NEURAL_MODELS) & (history['seed'] == SEED)]
curves = curves.sort_values(['model', 'epoch']).copy()
curves['elapsed_minutes'] = curves.groupby('model')['epoch_seconds'].cumsum() / 60
curves['rmse_smooth'] = curves.groupby('model')['validation_rmse'].transform(
    lambda values: values.rolling(SMOOTH_WINDOW, center=True, min_periods=1).mean(),
)
checkpoints = runs[runs['model'].isin(NEURAL_MODELS) & (runs['seed'] == SEED)]
checkpoints = checkpoints[['model', 'best_epoch']].rename(columns={'best_epoch': 'epoch'})
checkpoints = checkpoints.merge(curves, on=['model', 'epoch'], validate='one_to_one').set_index('model')


def plot_convergence(x_column, xlabel, title, name):
    fig, ax = plt.subplots(figsize=(11, 6))
    for model in NEURAL_MODELS:
        curve = curves[curves['model'] == model]
        best = checkpoints.loc[model]
        ax.plot(curve[x_column], curve['rmse_smooth'], label=LABELS[model],
                color=COLORS[model], linewidth=1.8)
        ax.scatter(best[x_column], best['validation_rmse'], marker='*', s=220,
                   color=COLORS[model], edgecolor='white', linewidth=1.1, zorder=5)
        if x_column == 'epoch':
            ax.annotate(f"Vòng {int(best['epoch'])}",
                        (best[x_column], best['validation_rmse']), xytext=(8, 8),
                        textcoords='offset points', ha='left', color=COLORS[model], fontsize=10)
    ax.set(title=title, xlabel=xlabel, ylabel='RMSE trên tập xác thực')
    ax.margins(y=0.15)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(alpha=0.18)
    ax.legend()
    fig.tight_layout()
    save_figure(fig, name)


plot_convergence('epoch', 'Vòng huấn luyện', 'Quá trình hội tụ theo vòng huấn luyện',
                 'convergence_checkpoints')


## 7. Hội tụ theo thời gian

In [ ]:
plot_convergence('elapsed_minutes', 'Thời gian huấn luyện và xác thực (phút)',
                 'Quá trình hội tụ theo thời gian', 'convergence_elapsed_time')


## 8. Thời gian trung bình mỗi vòng huấn luyện

In [ ]:
epoch_time = history[history['model'].isin(NEURAL_MODELS)]
epoch_time = epoch_time.groupby('model')['epoch_seconds'].mean().reindex(NEURAL_MODELS)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar([LABELS[model] for model in NEURAL_MODELS], epoch_time,
       color=[COLORS[model] for model in NEURAL_MODELS])
ax.set(title='Thời gian trung bình mỗi vòng huấn luyện',
       xlabel='Mô hình', ylabel='Thời gian huấn luyện và xác thực (giây)')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
save_figure(fig, 'training_seconds_mean')


## 9. Số lượng tham số


In [ ]:
parameters = runtime[['model', 'seed', 'parameters']].copy()
parameters['parameters'] = parameters['parameters'].astype('Int64')
parameters.to_csv(FIG_DIR / 'model_parameters.csv', index=False)
parameter_table = parameters.rename(columns={
    'model': 'Mô hình', 'seed': 'Mã khởi tạo', 'parameters': 'Số tham số',
})
parameter_table['Mô hình'] = parameter_table['Mô hình'].map(LABELS)
display(parameter_table.style.format({'Số tham số': '{:,.0f}'}, na_rep='—').hide(axis='index'))